# exp101_pf_candidate_ranker_or_nway_classifier train

Train-side audit for supervised PF/Beam candidate selection using the exp099 v2 multi-observation likelihood cache.

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from pf_candidate_ranker_or_nway_classifier import (
    build_required_columns,
    candidate_specs_from_config,
    find_artifact,
    run_pf_candidate_ranker_or_nway_classifier,
    DEFAULT_TRAIN_FEATURE_CACHE,
    DEFAULT_TRAIN_FEATURE_SCHEMA,
)

paths = ExperimentPaths()
config = load_config()
output_dir = paths.artifacts_dir
output_dir.mkdir(parents=True, exist_ok=True)

print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('parent:', get_nested(config, 'lineage.parent'))
print('strategy:', get_nested(config, 'validation.strategy'))
print('output_dir:', output_dir)

## 2. Input cache audit

In [ ]:
candidates = candidate_specs_from_config(config)
required_columns = build_required_columns(config, candidates)
cache_path = find_artifact(
    DEFAULT_TRAIN_FEATURE_CACHE,
    get_nested(config, 'data.exp099_train_feature_cache_local'),
)
schema_path = find_artifact(
    DEFAULT_TRAIN_FEATURE_SCHEMA,
    get_nested(config, 'data.exp099_train_feature_schema_local'),
)
header = pd.read_csv(cache_path, nrows=0).columns.tolist()
missing = [column for column in required_columns if column not in header]
print('cache_path:', cache_path)
print('schema_path:', schema_path)
print('required_columns:', len(required_columns))
print('missing_required_columns:', missing)
if missing:
    raise RuntimeError(f'Missing required input columns: {missing}')
preview_cols = [column for column in ['id', 'well', 'target', 'last_known_tvt', 'pf_ancc', 'beam_mean', 'likpf_mean', 'multiobs_score_gap'] if column in header]
pd.read_csv(cache_path, usecols=preview_cols, nrows=5)

## 3. Candidate and model plan

In [ ]:
candidate_plan = pd.DataFrame([{'candidate': spec.name, 'column': spec.column} for spec in candidates])
model_plan = pd.DataFrame([
    {'variant': 'likpf_mean_single', 'kind': 'baseline'},
    {'variant': 'multiobs_score_top1', 'kind': 'target_free_baseline'},
    {'variant': 'oracle', 'kind': 'upper_bound'},
    {'variant': 'lgb_multiclass', 'kind': 'supervised_nway_classifier'},
    {'variant': 'lgb_candidate_binary', 'kind': 'supervised_candidate_long_classifier'},
    {'variant': 'lgb_candidate_error_ranker', 'kind': 'supervised_candidate_long_error_ranker'},
])
display(candidate_plan)
display(model_plan)
print('n_folds:', get_nested(config, 'validation.n_folds'))
print('long max train rows per fold:', get_nested(config, 'ranker.long_models.max_train_rows_per_fold'))

## 4. Run OOF ranker audit

In [ ]:
summary = run_pf_candidate_ranker_or_nway_classifier(
    output_dir=output_dir,
    cache_path=get_nested(config, 'data.exp099_train_feature_cache_local'),
    schema_path=get_nested(config, 'data.exp099_train_feature_schema_local'),
    max_rows=get_nested(config, 'ranker.max_rows'),
)
summary_path = output_dir / 'exp101_pf_candidate_ranker_or_nway_classifier_summary.json'
print('summary_path:', summary_path)
print(json.dumps(summary.get('decision', {}), indent=2, sort_keys=True))

## 5. Metrics and artifacts

In [ ]:
metrics_path = output_dir / 'exp101_pf_candidate_ranker_or_nway_classifier_metrics.csv'
dist_path = output_dir / 'exp101_pf_candidate_ranker_or_nway_classifier_selection_distribution.csv'
bucket_path = output_dir / 'exp101_pf_candidate_ranker_or_nway_classifier_bucket_metrics.csv'
importance_path = output_dir / 'exp101_pf_candidate_ranker_or_nway_classifier_feature_importance_mean.csv'

metrics = pd.read_csv(metrics_path).sort_values('rmse_tvt')
display(metrics)
display(pd.read_csv(dist_path).head(30))
display(pd.read_csv(bucket_path).sort_values('rmse_tvt', ascending=False).head(20))
display(pd.read_csv(importance_path).head(30))